In [1]:
import torch
from torchvision import models
import torchvision.transforms.functional as transform
import PIL
import os
from tqdm import tqdm
import json
import numpy as np
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

In [2]:
# Загрузка первоначальной модели с предобученными весами
model = torch.hub.load("chenyaofo/pytorch-cifar-models", 'cifar100_resnet20', pretrained=True)

num_classes = 31

in_features = model.fc.in_features  # количество входных признаков для полносвязного слоя
model.fc = nn.Linear(in_features=in_features, out_features=num_classes, bias=True)

state_dict = torch.load('my_model.pth')
new_state_dict = {}
for key, value in state_dict.items():
    if key.startswith('1.'):
        new_key = key[2:]
    else:
        new_key = key

    if new_key in model.state_dict() and value.size() == model.state_dict()[new_key].size():
        new_state_dict[new_key] = value


model.load_state_dict(new_state_dict, strict=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(model)


Using cache found in /home/darya/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
/tmp/ipykernel_6635/2986525407.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  st

CifarResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias

In [3]:
# у него дал самый последний слой на 100 классов
model.fc

Linear(in_features=64, out_features=31, bias=True)

In [4]:
# тождественное преобразование
class Identify(torch.nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return x

model.fc = Identify()

In [5]:
height_width = 32
preprocess = transforms.Compose([
    transforms.Resize((height_width, height_width), interpolation=Image.LANCZOS),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5074, 0.4867, 0.4411], std=[0.2011, 0.1987, 0.2025])
])

In [ ]:
img_path = './content/pics'
PIL_img = Image.open(img_path).convert('RGB')
# new_PIL_img = transform.to_pil_image(tensor_img)

In [ ]:
pil_img = Image.open("/content/pics/Вербена прерийная.jpeg").convert('RGB')

In [ ]:
pil_img

In [ ]:
pil_img.shape # 3.224.224

AttributeError: 'Image' object has no attribute 'shape'

In [ ]:
img_transformed = preprocess(pil_img)
img_transformed.shape

torch.Size([3, 32, 32])

In [6]:
# модель -  врежим вычисления, а не обучения
model.eval()

CifarResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias

In [ ]:
# картинку - в батч
batch_img = img_transformed.unsqueeze(0)
batch_img.shape # 1.3.224.224

torch.Size([1, 3, 32, 32])

In [ ]:
with torch.no_grad():
  prediction = model(batch_img)

prediction.shape # [1, N] -массив

torch.Size([1, 64])

In [8]:
files = os.listdir('./pics')
len(files)

20

In [9]:
imgs = []

for f in tqdm(files):
  pil_img = PIL.Image.open(f'./pics/{f}').convert('RGB')
  # img = transform.to_tensor(pil_img)
  img_transformed = preprocess(pil_img)
  imgs.append(img_transformed)

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:01<00:00, 12.63it/s]


In [10]:
len(imgs)

20

In [11]:
index = []
for f in files:
  index.append(f.split('.')[0])

with open('index.json', 'w') as f:
  json.dump(index, f)

In [12]:
index[9]

'Бархатцы французские'

In [44]:
plant_name = 'Бархатцы французские'
print(index.index(plant_name))

9


In [13]:
imgs[0].shape #3.224.224

torch.Size([3, 32, 32])

In [14]:
imgs[0].unsqueeze(0).shape #1,3.224.224

torch.Size([1, 3, 32, 32])

In [15]:
batch = torch.vstack(tuple((im.unsqueeze(0) for im in imgs)))

batch.shape # N.3.224.224

torch.Size([20, 3, 32, 32])

In [17]:
%%time

with torch.no_grad():
  vects = model(batch)

vects.shape # torch.size(N, 2048)

CPU times: user 314 ms, sys: 0 ns, total: 314 ms
Wall time: 78.6 ms


torch.Size([20, 64])

In [18]:
vects_norm = vects / torch.Tensor.repeat(vects.norm(dim=1).unsqueeze(1), 1, 64)
vects_norm_np = vects_norm.numpy()

In [19]:
np.min(vects_norm_np)

np.float32(0.0)

In [20]:
np.max(vects_norm_np)

np.float32(0.38968983)

In [21]:
with open('./vects.npy', 'wb') as f:
  np.save(f, vects_norm_np)

In [22]:
vects_norm_np = vects_norm_np.astype(np.float16)
vects_norm_np_q = vects_norm_np * 128
vects_norm_np_q = vects_norm_np_q.astype(np.int8)

with open('./vects_q.npy', 'wb') as f:
  np.save(f, vects_norm_np_q)

## дальше - рекомендашки

In [23]:
with open('./vects_q.npy', 'rb') as f:
  vects = np.load(f)

with open('./index.json', 'rb') as f:
  index = json.load(f)

In [24]:
def get_similar(v, vects, n=10):
  scores = np.matmul(vects, v)
  scores = scores / 128
  top_similat_ind = (-scores).argsort()[:n]
  return {
      'similar_ind': list(top_similat_ind),
      'similar_scores': list(scores[top_similat_ind])
  }

In [25]:
def get_by_indexs(inds, index):
  f_names = []
  for i in inds:
    f_names.append(f'/pics/{index[i]}_0.jpg')
  return f_names

In [26]:
def get_sim_mean(viewed_ids, vects, n=10):
  v = np.zeros(64)
  viewed_ids = viewed_ids[::-1][:10]
  for i in viewed_ids[:]:
    v += vects[i]
  v /= len(viewed_ids)
  v /= np.linalg.norm(v)
  return filter_(get_similar(v, vects, n+len(viewed_ids)), viewed_ids)


In [27]:
def filter_(recs, viewed_ids):
  res = {
      'similar_ind': [],
      'similar_scores': []
  }
  for i in range(len(recs['similar_ind'])):
    if recs['similar_ind'][i] not in viewed_ids:
      res['similar_ind'].append(recs['similar_ind'][i])
      res['similar_scores'].append(recs['similar_scores'][i])
  return res

In [28]:
class NpEncoder(json.JSONEncoder):
  def default(self, obj):
    if isinstance(obj, np.integer):
      return int(obj)
    if isinstance(obj, np.floating):
      return float(obj)
    if isinstance(obj, np.ndarray):
      return obj.tolist()
    return super(NpEncoder, self).default(obj)


In [29]:
viewed_ids = [1, 4]

In [30]:
res = get_sim_mean(viewed_ids, vects)

In [31]:
for i in range(len(index)):
  print(i, index[i])

0 Вербена сиреневая
1 Бархатцы "Легион Чести"
2 Гиацинт махровый
3 Космея "Соната Кармин"
4 Космея Глория
5 Космея абрикосовая
6 Гиацинт "Голубая Звезда"
7 Вербена Королевский Красный
8 Бархатцы "Большая Золотая Утка"
9 Бархатцы французские
10 Гиацинт "Белая Жемчужина"
11 Вербена Темный Голубой
12 Вербена Супербена Бургунди
13 Гиацинт голландский
14 Космея "Конфетная Полоска"
15 Вербена прерийная
16 Бархатцы "Большая Оранжевая Утка"
17 Вербена персиковая
18 Гиацинт "Фондант"
19 Космея шоколадная


In [32]:
res['similar_ind']

[np.int64(9),
 np.int64(3),
 np.int64(5),
 np.int64(19),
 np.int64(7),
 np.int64(8),
 np.int64(14),
 np.int64(16),
 np.int64(10),
 np.int64(17)]

In [48]:
similar_ids = list(map(int, res['similar_ind']))
print(similar_ids[6])

14


In [33]:
res['similar_scores']

[np.float64(0.8917577406928082),
 np.float64(0.8517042433378892),
 np.float64(0.8409231769665235),
 np.float64(0.8239671964196078),
 np.float64(0.8015706158153157),
 np.float64(0.7864837984782962),
 np.float64(0.7766706916263411),
 np.float64(0.7663235381429869),
 np.float64(0.7420244164143361),
 np.float64(0.7103153976750252)]

In [34]:
file_names = get_by_indexs(res['similar_ind'], index)

# Теперь file_names будет содержать соответствующие названия
print(file_names)

['/pics/Бархатцы французские_0.jpg', '/pics/Космея "Соната Кармин"_0.jpg', '/pics/Космея абрикосовая_0.jpg', '/pics/Космея шоколадная_0.jpg', '/pics/Вербена Королевский Красный_0.jpg', '/pics/Бархатцы "Большая Золотая Утка"_0.jpg', '/pics/Космея "Конфетная Полоска"_0.jpg', '/pics/Бархатцы "Большая Оранжевая Утка"_0.jpg', '/pics/Гиацинт "Белая Жемчужина"_0.jpg', '/pics/Вербена персиковая_0.jpg']


In [35]:
json.dumps(res, cls=NpEncoder)

'{"similar_ind": [9, 3, 5, 19, 7, 8, 14, 16, 10, 17], "similar_scores": [0.8917577406928082, 0.8517042433378892, 0.8409231769665235, 0.8239671964196078, 0.8015706158153157, 0.7864837984782962, 0.7766706916263411, 0.7663235381429869, 0.7420244164143361, 0.7103153976750252]}'

In [46]:
res_ids = res['similar_ind']
json_output = json.dumps(res_ids, cls=NpEncoder)
print(json_output)
print(json_output[0])

[9, 3, 5, 19, 7, 8, 14, 16, 10, 17]
[
